# Модуль 4b — от MLP до nano-GPT

В [Модуле 4a](https://itrubnikov.github.io/Train_of_Thought/docs/modules/04a-micrograd/)
ты обучил **один** нейрон руками. Здесь поднимемся на пару ступенек:

1. соберём **MLP** (multilayer perceptron, многослойный перцептрон) на
   чистом NumPy — forward и backward напишем руками, как в 4a, только
   теперь не для одного числа, а для матриц;
2. перепишем **ту же** сеть на PyTorch и убедимся: код короче, а это
   те же forward/backward, только быстрые и на GPU;
3. (опционально, нужен GPU) обучим маленькую **CNN** на картинках
   CIFAR-10.

Полный гайд с картинками — в
[README этой папки](https://github.com/ITrubnikov/Train_of_Thought-homework/tree/main/notebooks/module-4b-nano-gpt).
Лекция — [Модуль 4b](https://itrubnikov.github.io/Train_of_Thought/docs/modules/04b-nano-gpt/).

Прогоняй сверху вниз. Места с `# TODO` — это домашка.

## Главная идея на пальцах

**Слой сети — это умножение матриц.** В 4a один нейрон считал
`y = tanh(w*x + b)`, где `w` и `x` — числа. Поставим рядом много
нейронов на тех же входах — получим **слой**. Теперь `x` — это вектор
(пачка чисел), `W` — табличка весов (матрица), и весь слой считается
одной операцией:

```text
h = tanh(X @ W1 + b1)   <- слой 1: умножение матриц + нелинейность
logits = h @ W2 + b2    <- слой 2: ещё одно умножение матриц
```

Значок `@` в Python — это **умножение матриц** (matrix multiply). Вся
разница с 4a в том, что локальные производные стали не числами, а
матрицами. Сам приём — тот же chain rule: `dL/da = dL/de * de/da`.

**Зачем это знать:** именно потому, что слой = умножение матриц, всё
обучение упирается в GPU (видеокарты умножают огромные матрицы тысячами
потоков сразу). Никакой другой причины «нужен GPU» нет.

---
# Часть 1. MLP на чистом NumPy

Никакого PyTorch. Только `numpy`, чтобы увидеть слои **как умножения
матриц** и backprop **как тот же chain rule из 4a**, только записанный
матрицами.

## Шаг 1 — игрушечный датасет: две спирали

Возьмём классическую игрушку — точки трёх классов, закрученные в
спирали. Прямой линией их не разделить, поэтому простой линейной модели
тут не хватит — нужна нелинейность (та самая `tanh`). Это маленький, но
честный тест: если сеть выучит спирали, значит слои и backprop собраны
правильно.

**Что где:** `X` — координаты точек (N строк по 2 числа), `y` — метка
класса (0, 1 или 2). Всё помещается в память, GPU не нужен.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)

def make_spirals(points_per_class=100, n_classes=3):
    N, K = points_per_class, n_classes
    X = np.zeros((N * K, 2))
    y = np.zeros(N * K, dtype=int)
    for k in range(K):
        idx = range(N * k, N * (k + 1))
        r = np.linspace(0.0, 1.0, N)                       # радиус
        t = np.linspace(k * 4, (k + 1) * 4, N) + np.random.randn(N) * 0.2  # угол
        X[idx] = np.c_[r * np.sin(t), r * np.cos(t)]
        y[idx] = k
    return X, y

X, y = make_spirals()
print("X:", X.shape, " y:", y.shape, " классов:", len(set(y)))

plt.scatter(X[:, 0], X[:, 1], c=y, s=12, cmap="brg")
plt.title("Три спирали — их линией не разделить")
plt.show()

## Шаг 2 — forward руками

Сеть из двух слоёв (один скрытый). На входе 2 числа (координаты), в
скрытом слое `H` нейронов, на выходе 3 числа (logits — по одному на
класс).

- `W1` имеет форму `(2, H)`: 2 входа -> H нейронов. Каждый **столбец** —
  веса одного нейрона из 4a.
- `tanh` — та же нелинейность, что в 4a. Без неё два слоя подряд
  схлопнулись бы в один линейный (см. ДЗ-1).
- `W2` имеет форму `(H, 3)`: H -> 3 класса.

`logits` — это «сырые» числа на выходе, ещё не вероятности. Превратим
их в вероятности на следующем шаге.

In [ ]:
H = 100  # нейронов в скрытом слое

# инициализируем веса маленькими случайными числами, смещения нулями
W1 = 0.1 * np.random.randn(2, H)
b1 = np.zeros((1, H))
W2 = 0.1 * np.random.randn(H, 3)
b2 = np.zeros((1, 3))

def forward(X):
    z1 = X @ W1 + b1        # слой 1: умножение матриц + смещение
    h = np.tanh(z1)         # нелинейность (как в 4a)
    logits = h @ W2 + b2    # слой 2: ещё одно умножение матриц
    return z1, h, logits

_, _, logits = forward(X)
print("logits:", logits.shape)   # (300, 3) — по 3 числа на каждую точку

## Шаг 3 — loss: softmax + cross-entropy

Два новых слова, оба простые.

- **softmax** превращает три выходных числа (logits) в три вероятности:
  большое число -> большая вероятность, и всё в сумме даёт 1. Просто
  способ сказать «модель на 80% уверена, что это класс 0».
- **cross-entropy** — это loss поверх вероятностей: одно число,
  «насколько уверенно модель ошиблась». Угадала уверенно — почти 0;
  уверенно села в лужу — большое число. Роль та же, что у квадрата
  ошибки в 4a, просто для классов вместо одного числа.

**Зачем:** loss — это и есть та «горка», по которой мы катимся вниз
градиентным спуском. Чем меньше loss, тем лучше сеть угадывает класс.

In [ ]:
def softmax(logits):
    # вычитаем максимум в строке — для численной устойчивости (без этого
    # exp от больших чисел переполняется). На результат это не влияет.
    z = logits - logits.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)

def cross_entropy(probs, y):
    N = y.shape[0]
    # берём вероятность, которую сеть дала ПРАВИЛЬНОМУ классу каждой точки
    correct_logprobs = -np.log(probs[range(N), y] + 1e-12)
    return correct_logprobs.mean()

probs = softmax(logits)
loss = cross_entropy(probs, y)
print("стартовый loss:", round(float(loss), 4))
# при 3 классах случайная сеть даёт примерно log(3) = 1.0986
print("loss случайной сети ~", round(float(np.log(3)), 4))

## Шаг 4 — backward руками (тот же chain rule, только матрицами)

Здесь самая мякотка. Считаем, как loss зависит от каждого веса, чтобы
знать, в какую сторону их крутить. Идём справа налево, ровно как в 4a:

1. У softmax + cross-entropy есть приятное свойство: градиент по logits
   получается очень простым — `probs`, из которого вычли 1 у правильного
   класса (и поделили на число точек). Это `dlogits`.
2. Дальше `dlogits` течёт назад через `@ W2`, через `tanh`, через `@ W1`
   — на каждом шаге умножаясь на локальную производную. Локальная
   производная `tanh` — та же, что в 4a: `1 - tanh^2`.

`dW1, dW2, db1, db2` — это «наклоны» loss по каждому весу. В PyTorch их
посчитает `loss.backward()` за тебя; здесь разворачиваем руками, чтобы
увидеть, что магии внутри нет.

In [ ]:
def backward(X, y, z1, h, probs):
    N = y.shape[0]

    # 1) градиент по logits (свойство softmax+CE: probs минус 1 у верного класса)
    dlogits = probs.copy()
    dlogits[range(N), y] -= 1
    dlogits /= N

    # 2) слой 2: logits = h @ W2 + b2
    dW2 = h.T @ dlogits
    db2 = dlogits.sum(axis=0, keepdims=True)

    # 3) назад через tanh: dh, затем dz1 = dh * (1 - tanh^2)
    dh = dlogits @ W2.T
    dz1 = dh * (1 - h**2)           # та же производная tanh, что в 4a

    # 4) слой 1: z1 = X @ W1 + b1
    dW1 = X.T @ dz1
    db1 = dz1.sum(axis=0, keepdims=True)

    return dW1, db1, dW2, db2

# проверим, что формы градиентов совпадают с формами весов
z1, h, logits = forward(X)
probs = softmax(logits)
dW1, db1, dW2, db2 = backward(X, y, z1, h, probs)
print("dW1", dW1.shape, "== W1", W1.shape)
print("dW2", dW2.shape, "== W2", W2.shape)

## Шаг 5 — цикл обучения (тот же, что в 4a)

`forward -> loss -> backward -> шаг`. Один в один цикл из 4a, только
вместо `w.data -= lr * w.grad` для одного числа мы обновляем целые
матрицы весов.

`lr` (learning rate) — размер шага. Маленький — учимся медленно;
большой — можем «перепрыгнуть» минимум. Подберём на глаз.

In [ ]:
# заново инициализируем веса, чтобы цикл можно было перезапускать с нуля
np.random.seed(0)
W1 = 0.1 * np.random.randn(2, H)
b1 = np.zeros((1, H))
W2 = 0.1 * np.random.randn(H, 3)
b2 = np.zeros((1, 3))

lr = 1.0          # для этой игрушки можно крупный шаг
losses = []

for step in range(2000):
    # forward
    z1, h, logits = forward(X)
    probs = softmax(logits)
    loss = cross_entropy(probs, y)
    losses.append(loss)

    # backward
    dW1, db1, dW2, db2 = backward(X, y, z1, h, probs)

    # шаг градиентного спуска: против градиента
    W1 -= lr * dW1
    b1 -= lr * db1
    W2 -= lr * dW2
    b2 -= lr * db2

    if step % 200 == 0:
        pred = logits.argmax(axis=1)
        acc = (pred == y).mean()
        print(f"step {step:4d}  loss={loss:.4f}  acc={acc:.3f}")

print("итоговая accuracy:", round(float((forward(X)[2].argmax(1) == y).mean()), 3))

## Шаг 6 — граница решения (decision boundary)

Закрасим плоскость по тому, какой класс предсказывает сеть в каждой
точке. Если видим три закрученных «лепестка» — сеть выучила спирали, и
наш ручной backprop работает.

In [ ]:
def plot_boundary(predict_fn, X, y, title):
    h_step = 0.02
    x_min, x_max = X[:, 0].min() - 0.3, X[:, 0].max() + 0.3
    y_min, y_max = X[:, 1].min() - 0.3, X[:, 1].max() + 0.3
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h_step),
                         np.arange(y_min, y_max, h_step))
    grid = np.c_[xx.ravel(), yy.ravel()]
    Z = predict_fn(grid).reshape(xx.shape)
    plt.contourf(xx, yy, Z, alpha=0.4, cmap="brg")
    plt.scatter(X[:, 0], X[:, 1], c=y, s=12, cmap="brg", edgecolors="k", linewidths=0.2)
    plt.title(title)
    plt.show()

def numpy_predict(grid):
    return forward(grid)[2].argmax(axis=1)

plot_boundary(numpy_predict, X, y, "NumPy MLP — граница решения")

---
# Часть 2. Тот же MLP на PyTorch

Теперь соберём **ту же** сеть, но на PyTorch. Главная мысль модуля:
**PyTorch — это твой micrograd, только быстрый.** Те же forward/backward,
только на тензорах и (при желании) на GPU.

| Твой NumPy / micrograd | PyTorch | то же самое? |
| --- | --- | --- |
| матрицы `W1, W2` | `nn.Linear` | да, веса внутри слоя |
| `forward(...)` руками | `model(x)` | да |
| `backward(...)` руками | `loss.backward()` | один в один |
| `W -= lr * dW` | `optimizer.step()` | да |
| обнуление градиентов | `optimizer.zero_grad()` | да (как в 4a!) |

В Colab PyTorch уже установлен. Локально: `pip install torch`.

In [ ]:
import torch
import torch.nn as nn

# те же данные, но как тензоры PyTorch
Xt = torch.tensor(X, dtype=torch.float32)
yt = torch.tensor(y, dtype=torch.long)
print("torch версии:", torch.__version__)
print("Xt:", tuple(Xt.shape), " yt:", tuple(yt.shape))

## Шаг 7 — модель в три строки

`nn.Sequential` — это просто «слои подряд», как наш `forward`. Сравни с
Частью 1: `nn.Linear(2, H)` это `X @ W1 + b1`, `nn.Tanh()` это
`np.tanh`, `nn.Linear(H, 3)` это `h @ W2 + b2`. Один в один, но PyTorch
сам заведёт и веса, и backward.

In [ ]:
torch.manual_seed(0)

model = nn.Sequential(
    nn.Linear(2, H),    # = X @ W1 + b1
    nn.Tanh(),          # = np.tanh
    nn.Linear(H, 3),    # = h @ W2 + b2
)

# CrossEntropyLoss внутри сам делает softmax + cross-entropy (наш Шаг 3)
loss_fn = nn.CrossEntropyLoss()
# SGD = тот же шаг против градиента, что мы писали руками
optimizer = torch.optim.SGD(model.parameters(), lr=1.0)

print(model)

## Шаг 8 — цикл обучения на PyTorch

Тот же `forward -> loss -> backward -> шаг`. Сравни построчно с Шагом 5:
- `model(Xt)` вместо ручного `forward`;
- `loss.backward()` вместо ручного `backward` (PyTorch посчитал все
  `dW` за нас через autograd);
- `optimizer.step()` вместо ручного `W -= lr * dW`;
- `optimizer.zero_grad()` — то самое обнуление градиентов из 4a (там
  стояло `w.grad = 0`). Забудешь — градиенты накопятся, как мы видели.

In [ ]:
torch_losses = []

for step in range(2000):
    logits_t = model(Xt)            # forward
    loss_t = loss_fn(logits_t, yt)  # loss (softmax+CE внутри)

    optimizer.zero_grad()           # обнулить градиенты (как w.grad=0 в 4a)
    loss_t.backward()               # backward — autograd считает все dW
    optimizer.step()                # шаг: против градиента

    torch_losses.append(loss_t.item())
    if step % 200 == 0:
        acc = (logits_t.argmax(1) == yt).float().mean().item()
        print(f"step {step:4d}  loss={loss_t.item():.4f}  acc={acc:.3f}")

final_acc = (model(Xt).argmax(1) == yt).float().mean().item()
print("итоговая accuracy (PyTorch):", round(final_acc, 3))

## Шаг 9 — сравним границы решения

Нарисуем границу решения PyTorch-версии и сравним кривые loss обеих
сетей. Должно совпасть: код стал короче, результат — тот же.

In [ ]:
@torch.no_grad()                       # для предсказания градиенты не нужны
def torch_predict(grid):
    g = torch.tensor(grid, dtype=torch.float32)
    return model(g).argmax(1).numpy()

plot_boundary(torch_predict, X, y, "PyTorch MLP — граница решения")

plt.plot(losses, label="NumPy (руками)")
plt.plot(torch_losses, label="PyTorch")
plt.xlabel("шаг")
plt.ylabel("loss")
plt.title("Loss: руками vs PyTorch — один и тот же спуск")
plt.legend()
plt.show()

---
# Домашка (часть 1): MLP на NumPy и на PyTorch

Соответствует HomeworkBlock «MLP на NumPy -> тот же MLP на PyTorch»
из лекции. Цель — прочувствовать, что PyTorch это твой ручной backprop,
только быстрый.

## ДЗ-1. Убери нелинейность и посмотри, что сломается

Возьми PyTorch-модель и **убери** `nn.Tanh()` между слоями (оставь два
`nn.Linear` подряд). Переобучи и нарисуй границу решения.

Что произойдёт с accuracy на спиралях и почему? Подсказка: два линейных
слоя подряд (`X @ W1 @ W2`) — это **один** линейный слой
(`X @ (W1 @ W2)`), а прямой линией спирали не разделить.

Запиши вывод одним абзацем в markdown-ячейке ниже.

In [ ]:
# TODO ДЗ-1: модель БЕЗ tanh (два nn.Linear подряд), обучи и нарисуй границу.
#
# torch.manual_seed(0)
# linear_model = nn.Sequential(
#     nn.Linear(2, H),
#     nn.Linear(H, 3),     # нет nn.Tanh() между ними
# )
# ... тот же цикл обучения ...
# ... plot_boundary(...) ...
#
# Ниже добавь markdown-ячейку: почему без нелинейности сеть не справляется.


## ДЗ-2. Добавь третий слой

Верни `tanh` и сделай сеть глубже: `Linear -> Tanh -> Linear -> Tanh ->
Linear`. Стало лучше, так же или начало мешать (обучается дольше,
accuracy не растёт)? Проверь экспериментом, не угадывай — запиши
наблюдение одним абзацем.

In [ ]:
# TODO ДЗ-2: сеть с ДВУМЯ скрытыми слоями (Linear-Tanh-Linear-Tanh-Linear).
#             Обучи, сравни accuracy и время с двухслойной версией.


---
# Часть 3 (опционально). CNN на CIFAR-10

> ВНИМАНИЕ: этой части **нужен GPU**. На CPU обучение займёт часы.
> В Colab включи бесплатный T4: `Среда выполнения -> Сменить среду
> выполнения -> T4 GPU`. Если GPU нет — спокойно пропусти эту часть,
> зачёт по домашке даёт часть 1.

**Зачем это здесь.** До сих пор каждый вход был связан с каждым нейроном
(полносвязный MLP). Для картинок это расточительно: важные вещи (край,
угол, пятно) сидят в **соседних** пикселях. **CNN** (convolutional
neural network, свёрточная сеть) встраивает эту подсказку прямо в
архитектуру — маленькое окно скользит по картинке и смотрит только на
соседей. Это пример **inductive bias**: форма сети заранее «знает», что
важно, и потому учится быстрее и на меньших данных.

Проверь, что GPU доступен — следующая ячейка должна напечатать
`cuda`.

In [ ]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print("устройство:", device)
if device == "cpu":
    print("GPU не найден. Включи T4 в Colab или пропусти Часть 3 —"
          " зачёт даёт Часть 1.")

## Шаг 10 — загрузка CIFAR-10

CIFAR-10 — 60 000 цветных картинок 32x32 по 10 классам (самолёт,
машина, кот, собака...). `torchvision` скачает датасет сам.
`DataLoader` нарезает его на батчи (пачки картинок) — так обучение идёт
по чуть-чуть, а не всё разом в память.

In [ ]:
import torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader

transform = T.Compose([
    T.ToTensor(),
    T.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),  # привести пиксели к ~[-1, 1]
])

train_set = torchvision.datasets.CIFAR10(root="./data", train=True,
                                         download=True, transform=transform)
test_set = torchvision.datasets.CIFAR10(root="./data", train=False,
                                        download=True, transform=transform)

train_loader = DataLoader(train_set, batch_size=128, shuffle=True, num_workers=2)
test_loader = DataLoader(test_set, batch_size=256, shuffle=False, num_workers=2)

classes = ("plane", "car", "bird", "cat", "deer",
           "dog", "frog", "horse", "ship", "truck")
print("train:", len(train_set), " test:", len(test_set))

## Шаг 11 — маленькая CNN

Свёрточный слой `nn.Conv2d` — это и есть то самое скользящее окно.
`MaxPool2d` ужимает картинку вдвое (берёт максимум в окошке 2x2),
оставляя самое заметное. После пары таких блоков разворачиваем всё в
вектор и заканчиваем обычными `nn.Linear` — теми же, что в Части 2.

То есть CNN это не новая вселенная, а тот же MLP, которому спереди
приделали свёртки-«глаза».

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)   # окно 3x3
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)                            # ужать вдвое
        self.fc1 = nn.Linear(64 * 8 * 8, 128)                    # дальше обычный MLP
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))   # 32x32 -> 16x16
        x = self.pool(F.relu(self.conv2(x)))   # 16x16 -> 8x8
        x = x.flatten(1)                        # развернуть в вектор
        x = F.relu(self.fc1(x))
        return self.fc2(x)

cnn = SmallCNN().to(device)
print(cnn)

## Шаг 12 — обучение CNN

Тот же цикл `forward -> loss -> backward -> шаг`, что и для MLP. Новое
только одно: данные идут **батчами** из `DataLoader`, и каждый батч мы
перекидываем на GPU через `.to(device)`. Пара эпох (проходов по всем
данным) на T4 — это минуты, а не часы.

In [ ]:
import torch.optim as optim

loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(cnn.parameters(), lr=1e-3)  # Adam — продвинутый SGD

EPOCHS = 5
for epoch in range(EPOCHS):
    cnn.train()
    running = 0.0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)   # на GPU

        logits = cnn(imgs)                 # forward
        loss = loss_fn(logits, labels)     # loss

        optimizer.zero_grad()              # обнулить градиенты
        loss.backward()                    # backward (autograd)
        optimizer.step()                   # шаг

        running += loss.item()
    print(f"эпоха {epoch + 1}/{EPOCHS}  средний loss={running / len(train_loader):.4f}")

print("обучение CNN закончено")

## Шаг 13 — точность на тесте

Считаем accuracy на картинках, которых сеть не видела. Для маленькой
CNN за 5 эпох нормально получить ~65-72%. Случайная угадайка дала бы
10% (10 классов), так что сеть явно что-то выучила.

In [ ]:
cnn.eval()
correct = total = 0
with torch.no_grad():
    for imgs, labels in test_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        preds = cnn(imgs).argmax(1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

print(f"точность на тесте: {100 * correct / total:.1f}%")

## Домашка (часть 2, опционально): CNN vs MLP на CIFAR-10

Соответствует HomeworkBlock «CNN на CIFAR-10 (опционально, на GPU)»
из лекции.

**ДЗ-3.** Доведи CNN выше до accuracy ~70% (поиграй с числом эпох или
ещё одним свёрточным слоем).

**ДЗ-4.** Замени CNN на **MLP такого же примерно размера** (разверни
картинку `3*32*32 = 3072` в вектор и прогони через пару `nn.Linear`).
Обучи столько же эпох. Насколько MLP хуже на тех же картинках — и почему
свёртки выигрывают? (подсказка: inductive bias — окно заранее знает про
соседние пиксели, MLP вынужден учить это с нуля).

Запиши вывод одним абзацем.

In [ ]:
# TODO ДЗ-4: MLP-бейзлайн на тех же картинках для сравнения с CNN.
#
# class SmallMLP(nn.Module):
#     def __init__(self):
#         super().__init__()
#         self.net = nn.Sequential(
#             nn.Flatten(),              # 3x32x32 -> 3072
#             nn.Linear(3072, 256), nn.ReLU(),
#             nn.Linear(256, 10),
#         )
#     def forward(self, x):
#         return self.net(x)
#
# Обучи так же, как CNN, и сравни accuracy на тесте.


---
# Что сдать

**Обязательно (часть 1):**

- Публичная ссылка на этот Colab-ноутбук — запускается сверху вниз без
  правок (части 1 и 2 целиком).
- Скриншоты границы решения (decision boundary) для NumPy- и
  PyTorch-версии.
- Один абзац: почему без нелинейности (`tanh`) сеть не справляется со
  спиралями.

**Критерий приёма:** PyTorch-версия даёт ту же accuracy, что NumPy
(близко к 1.0 на спиралях); ты можешь объяснить роль нелинейности.

**Опционально (часть 3, нужен GPU):** скриншот точности CNN на тесте и
абзац «почему свёртки выигрывают у MLP на картинках».

В чат как `[Модуль 4b, ДЗ] {ссылка}`.